# <font color="steelblue">28. Intervalo de confianza para una media y tamaño de muestra</font>

*Unidad 5. Conceptos básicos de inferencia estadística* · Sesión teórica 28 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv"]
PREGUNTAS = ["q_P28_1.json", "q_P28_2.json", "q_P28_3.json", "q_P28_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«Los españoles dedican de media 54 minutos al día a las noticias.» ¿Con qué precisión? Y si un medio quiere encargar una encuesta con un margen de ±2 puntos, ¿cuántas entrevistas debe pagar? Este cuaderno responde a las dos preguntas: el **intervalo de confianza para una media** y el **cálculo del tamaño de muestra**.

**Qué necesitas saber antes**: el cuaderno 27.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular un intervalo de confianza para una media con la t de Student.
2. Comparar intervalos de varios grupos en un gráfico.
3. Calcular el tamaño de muestra necesario para un margen de error dado.
4. Entender por qué el tamaño de la población apenas influye.

## <font color='steelblue'>Contenidos</font>

1. Intervalo para una media
2. Comparar grupos con sus intervalos
3. ¿Cuántas entrevistas necesito?
4. ¿Y el tamaño de la población?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Intervalo para una media</font>

Para una media, el intervalo al 95 % es:

$$\bar{x} \pm t_{n-1} \cdot \frac{s}{\sqrt{n}}$$

donde $t_{n-1}$ es un valor de la **t de Student** (muy parecido a 1,96 cuando $n$ es grande), porque además de la media estimamos la desviación típica.

In [ ]:
enc = pd.read_csv("encuesta_medios.csv")
minutos = enc["minutos_noticias_dia"].dropna()
n = len(minutos)
media, s = minutos.mean(), minutos.std()
t = stats.t.ppf(0.975, df=n - 1)
ee = s / np.sqrt(n)
print(f"n = {n}, media = {media:.1f} min, s = {s:.1f}, t = {t:.3f}")
print(f"IC 95 %: entre {media - t * ee:.1f} y {media + t * ee:.1f} minutos")

Con `scipy` se obtiene directamente:

In [ ]:
stats.t.interval(0.95, df=n - 1, loc=media, scale=ee)

**Cómo leerlo**: con 1.164 respuestas la media se conoce con una precisión de unos ±1,7 minutos. Fíjate en que el intervalo es para la **media**, no para cada persona: las personas varían muchísimo más.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P28_1.json")

## <font color='steelblue'>2. Comparar grupos con sus intervalos</font>

**Qué vamos a hacer**: calcular la media de minutos y su intervalo para cada medio principal y dibujarlos con barras de error.

In [ ]:
def ic_media(x, confianza=0.95):
    x = x.dropna()
    ee = x.std() / np.sqrt(len(x))
    t = stats.t.ppf(1 - (1 - confianza) / 2, df=len(x) - 1)
    return pd.Series({"n": len(x), "media": x.mean(), "margen": t * ee})

tabla = enc.groupby("medio_principal")["minutos_noticias_dia"].apply(ic_media).unstack().sort_values("media")
tabla.round(1)

In [ ]:
plt.errorbar(tabla["media"], tabla.index, xerr=tabla["margen"], fmt="o", color="steelblue", capsize=4)
plt.xlabel("Minutos al día con noticias (media e IC 95 %)")
plt.title("Consumo de noticias según el medio principal (encuesta simulada)")
plt.show()

Los grupos pequeños (radio, prensa en papel) tienen intervalos más anchos. Si los intervalos de dos grupos se solapan mucho, no podemos asegurar que sus medias sean distintas.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P28_2.json")

## <font color='steelblue'>3. ¿Cuántas entrevistas necesito?</font>

Despejando $n$ en la fórmula del margen de una proporción:

$$n = \frac{z^2\, p(1-p)}{e^2}$$

con $p = 0{,}5$ si no se sabe nada (peor caso) y $e$ el margen deseado en tanto por uno.

In [ ]:
def tamano_muestra(margen, p=0.5, confianza=0.95):
    z = stats.norm.ppf(1 - (1 - confianza) / 2)
    return int(np.ceil(z**2 * p * (1 - p) / margen**2))

for e in [0.05, 0.03, 0.02, 0.01]:
    print(f"Margen ±{e * 100:.0f} puntos: {tamano_muestra(e):>6} entrevistas")

**Cómo leerlo**: pasar de ±3 a ±1 punto multiplica las entrevistas por 9. Por eso casi ninguna encuesta pública baja de ±2 puntos: sería carísima.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P28_3.json")

## <font color='steelblue'>4. ¿Y el tamaño de la población?</font>

Sorprendentemente, el tamaño de la población **casi no importa** salvo que la muestra sea una parte grande de ella. Una encuesta de 1.000 personas tiene prácticamente el mismo margen en Castellón que en toda España. Se ajusta con el **factor de corrección para poblaciones finitas**:

$$n_{ajustado} = \frac{n}{1 + \frac{n - 1}{N}}$$

In [ ]:
n0 = tamano_muestra(0.03)
for N in [500, 5_000, 50_000, 5_000_000, 48_000_000]:
    print(f"Población {N:>11,}: {int(np.ceil(n0 / (1 + (n0 - 1) / N))):>5} entrevistas para ±3 puntos")

Solo en poblaciones pequeñas (los socios de un club, la plantilla de una empresa) la corrección reduce de forma notable las entrevistas necesarias.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P28_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Calcula el intervalo al 95 % para la confianza media en los medios.

<details><summary>Ver solución</summary>

```python
ic_media(enc["confianza_medios"])
```
</details>

### **Ejercicio 2** *(sección 1)*

Calcula el mismo intervalo al 99 %. ¿Es más ancho o más estrecho?

<details><summary>Ver solución</summary>

```python
ic_media(enc["confianza_medios"], 0.99)
```
</details>

### **Ejercicio 3** *(sección 2)*

Haz el gráfico de medias con intervalos de la confianza en los medios según el nivel de estudios.

<details><summary>Ver solución</summary>

```python
t2 = enc.groupby("estudios")["confianza_medios"].apply(ic_media).unstack()
plt.errorbar(t2["media"], t2.index, xerr=t2["margen"], fmt="o", capsize=4)
plt.show()
```
</details>

### **Ejercicio 4** *(sección 3)*

Un diario quiere estimar el porcentaje de lectores satisfechos con un margen de ±4 puntos. Cree que ronda el 80 %. ¿Cuántas encuestas necesita? ¿Y si no sabe nada (p = 0,5)?

<details><summary>Ver solución</summary>

```python
print(tamano_muestra(0.04, p=0.8), tamano_muestra(0.04))
```
</details>

### **Ejercicio 5** *(sección 4)*

La plantilla de un grupo de comunicación tiene 1.200 personas. ¿Cuántas hay que encuestar para un margen de ±5 puntos?

<details><summary>Ver solución</summary>

```python
n0 = tamano_muestra(0.05)
int(np.ceil(n0 / (1 + (n0 - 1) / 1200)))
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- **IC para una media**: $\bar{x} \pm t \cdot s/\sqrt{n}$; con muestras grandes $t \approx 1{,}96$.
- Las **barras de error** permiten comparar las medias de varios grupos con su incertidumbre.
- **Tamaño de muestra** para una proporción: $n = z^2 p(1-p)/e^2$; usa $p = 0{,}5$ si no sabes nada.
- Dividir el margen por 2 exige **cuadruplicar** la muestra.
- El tamaño de la población **apenas influye** salvo que sea pequeña (corrección para poblaciones finitas).

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **t de Student** | Distribución que se usa en lugar de la normal cuando se estima la desviación típica. |
| **Barras de error** | Segmentos que representan el intervalo de confianza en un gráfico. |
| **Tamaño de muestra** | Número de casos necesario para una precisión dada. |
| **Población finita** | Población pequeña en la que la muestra es una parte apreciable. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 29 aprenderás la lógica de los contrastes de hipótesis y el famoso p-valor.

**Siguiente cuaderno**: [29. Contrastes de hipótesis: la lógica del p-valor](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P29_contrastes_de_hipotesis.ipynb)